# 01 — DEAP data exploration

Goal: verify the downloaded dataset before applying preprocessing or training a model.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
from eeg_emotion.data import available_subject_ids, load_deap_subject
from eeg_emotion.preprocessing import binary_emotion_labels

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'deap'
subject_ids = available_subject_ids(DATA_DIR)
print(f'Found {len(subject_ids)} subjects: {subject_ids}')
assert subject_ids, 'No DEAP files found. Copy s01.dat ... s32.dat into data/raw/deap.'

In [ ]:
subject = load_deap_subject(DATA_DIR, subject_ids[0])
print('EEG shape:', subject.eeg.shape)
print('Labels shape:', subject.labels.shape)
print('EEG dtype:', subject.eeg.dtype)
print('All values finite:', np.isfinite(subject.eeg).all())
assert subject.eeg.shape[1] == 32

In [ ]:
valence = binary_emotion_labels(subject.labels, target='valence', threshold=5.0)
arousal = binary_emotion_labels(subject.labels, target='arousal', threshold=5.0)
print('Valence [low, high]:', np.bincount(valence, minlength=2))
print('Arousal [low, high]:', np.bincount(arousal, minlength=2))

In [ ]:
sampling_rate = 128
baseline_samples = 3 * sampling_rate
seconds_to_plot = 5
signal = subject.eeg[0, 0, baseline_samples:baseline_samples + seconds_to_plot * sampling_rate]
time = np.arange(signal.size) / sampling_rate

plt.figure(figsize=(12, 3))
plt.plot(time, signal, linewidth=0.8)
plt.xlabel('Time (seconds)')
plt.ylabel('EEG amplitude')
plt.title(f'Subject {subject.subject_id:02d}, trial 0, EEG channel 0')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Before continuing

Confirm that all 32 subject files are present, shapes are consistent, values are finite, and labels are sensible. The next notebook will create the participant-level split and 4-second windows.